In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans,MiniBatchKMeans
from pathlib import Path
import pickle

In [ ]:
def generate_clustered_heatmaps(
    df,
    features,
    save_dir,
    label,
    n_clusters,
    group_size,
    max_windows=None,
    scaler=None,
    random_sampling=False,
    kmeans_model=None,
    random_state=42
):
    df = df.copy()
    df = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

    df['original_index'] = np.arange(len(df))

    # Scale
    if scaler is None:
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(df[features])
    else:
        scaled = scaler.transform(df[features])
    
    df_scaled = pd.DataFrame(scaled, columns=features)
    df_scaled['original_index'] = df['original_index']

    #training set
    if kmeans_model is None:
      
        if len(df_scaled) > 100000:
            kmeans_model = MiniBatchKMeans(
                n_clusters=n_clusters, random_state=42,
                batch_size=2048, n_init=3
            )
        else:
            kmeans_model = KMeans(
                n_clusters=n_clusters, random_state=42, n_init=10
            )

        df_scaled['cluster'] = kmeans_model.fit_predict(df_scaled[features])

    else:
        #validation set
        df_scaled['cluster'] = kmeans_model.predict(df_scaled[features])


    #sort by cluster and time
    df_sorted = df_scaled.sort_values(by=['cluster', 'original_index'],ascending=[True, True]).reset_index(drop=True)


    #windows
    out_dir = Path(save_dir) / label
    out_dir.mkdir(parents=True, exist_ok=True)

    n = len(df_sorted)
    n_complete = n // group_size
    starts_all = [i * group_size for i in range(n_complete)]

    if max_windows and n_complete > max_windows:
        rng = np.random.RandomState(random_state)
        idx = rng.choice(n_complete, max_windows, replace=False)
        idx.sort()  
        starts = [starts_all[i] for i in idx]
    else:
        starts = starts_all

    print(f"Generating {len(starts)} windows")
    
    #heatmap gen
    for count, start in enumerate(starts, 1):
        end = start + group_size
        group = df_sorted.iloc[start:end][features]
        heatmap_len = len(group)
        min_required = int(group_size * 0.5)

        if heatmap_len < min_required:
            continue

        if heatmap_len < group_size:
            pad = np.zeros((group_size - heatmap_len, len(features)))
            heatmap = np.vstack([group.values, pad])
        else:
            heatmap = group.values

        fig, ax = plt.subplots(figsize=(5, 6))
        ax.imshow(
            heatmap, cmap="viridis", aspect="auto",
            interpolation="nearest", vmin=0, vmax=1
        )
        ax.axis("off")

        fname = out_dir / f"{label}_window_{count:04d}.png"
        plt.savefig(fname, bbox_inches="tight", pad_inches=0, dpi=120)
        plt.close(fig)

    print(f"Saved {len(starts)} heatmaps to {out_dir}\n")
    return scaler, kmeans_model


In [3]:
def split_flow_df(csv_path, features, val_fraction=0.1):
    df = pd.read_csv(csv_path)
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    n = len(df)
    val_size = int(n * val_fraction)

    df_train = df.iloc[:-val_size]
    df_val   = df.iloc[-val_size:]

    return df_train[features], df_val[features]

In [8]:
if __name__ == "__main__":
    features = [
        'Protocol', 'Flow Duration',
        'Flow Bytes/s', 'Flow Packets/s', 'Down/Up Ratio',
        'Total Fwd Packet', 'Total Bwd packets',
        'Fwd Header Length', 'Bwd Header Length',
        'FWD Init Win Bytes', 'Bwd Init Win Bytes',
        'Fwd Seg Size Min', 'Fwd Segment Size Avg',
        'Subflow Fwd Bytes', 'Subflow Bwd Bytes',
        'Packet Length Min', 'Packet Length Mean',
        'Packet Length Max', 'Packet Length Std',
        'Packet Length Variance', 'Average Packet Size',
        'Fwd IAT Mean', 'Bwd IAT Mean',
        'Fwd IAT Total', 'Bwd IAT Total',
        'Flow IAT Mean',
        'SYN Flag Count', 'ACK Flag Count',
        'PSH Flag Count', 'RST Flag Count', 'FIN Flag Count'
    ]


    #split to train/val - 90/10 
    benign_train, benign_val = split_flow_df(r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\Benign_train.pcap_Flow.csv", features)
    ddos_con_train, ddos_con_val = split_flow_df(r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\MQTT-DDoS-Connect_Flood_train.pcap_Flow.csv", features)
    ddos_pub_train, ddos_pub_val = split_flow_df(r"C:\Users\t0eur\OneDrive\Documents\CiCIOMT Data\flows\MQTT-DDoS-Publish_Flood_train.pcap_Flow.csv", features)

    #global scaler on train
    df_train_all = pd.concat([benign_train, ddos_con_train, ddos_pub_train])
    scaler = MinMaxScaler().fit(df_train_all)


    #train   
    _, benign_kmeans = generate_clustered_heatmaps(
        df=benign_train,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="benign",
        n_clusters=20,
        group_size=30, 
        scaler=scaler,
        kmeans_model=None
    )
    with open('benign_kmeans_K20.pkl', 'wb') as f:
        pickle.dump(benign_kmeans, f)
    with open('global_scaler.pkl', 'wb') as f: 
        pickle.dump(scaler, f)
    print("Benign cluster model saved!")

    _, ddos_con_kmeans = generate_clustered_heatmaps(
        df=ddos_con_train,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="ddos_connect",
        n_clusters=19,
        group_size=30,
        max_windows=1500,
        scaler=scaler,
        random_sampling=True,
        kmeans_model=None
    )
    with open('connect_kmeans_K15.pkl', 'wb') as f:
        pickle.dump(ddos_con_kmeans, f)
    print("Connect cluster model saved!")

    _, ddos_pub_kmeans = generate_clustered_heatmaps(
        df=ddos_pub_train,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="ddos_publish",
        n_clusters=20,
        group_size=30,
        max_windows=1500,
        scaler=scaler,
        random_sampling=True,
        kmeans_model=None
    )
    with open('publish_kmeans_K8.pkl', 'wb') as f:
        pickle.dump(ddos_pub_kmeans, f)
    print("Publish cluster model saved!")

    #validation
    generate_clustered_heatmaps(
        df=benign_val,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="benign_val",
        n_clusters=20,
        group_size=30,
        scaler=scaler,
        kmeans_model=benign_kmeans
    )

    generate_clustered_heatmaps(
        df=ddos_con_val,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="ddos_connect_val",
        n_clusters=19,
        group_size=30,
        scaler=scaler,
        max_windows=300,
        random_sampling=True,
        kmeans_model=ddos_con_kmeans
    )

    generate_clustered_heatmaps(
        df=ddos_pub_val,
        features=features,
        save_dir=r"D:\clustered_heatmaps_thursday_0212",
        label="ddos_publish_val",
        n_clusters=20,
        group_size=30,
        scaler=scaler,
        max_windows=300,
        random_sampling=True,
        kmeans_model=ddos_pub_kmeans
    )


Generating 1069 windows
Saved 1069 heatmaps to D:\clustered_heatmaps_thursday_0212\benign

Benign cluster model saved!


C:\Users\t0eur\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1952: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=8
  warnings.warn(


Generating 1500 windows
Saved 1500 heatmaps to D:\clustered_heatmaps_thursday_0212\ddos_connect

Connect cluster model saved!


C:\Users\t0eur\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1952: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=8
  warnings.warn(


Generating 1500 windows
Saved 1500 heatmaps to D:\clustered_heatmaps_thursday_0212\ddos_publish

Publish cluster model saved!
Generating 118 windows
Saved 118 heatmaps to D:\clustered_heatmaps_thursday_0212\benign_val

Generating 300 windows
Saved 300 heatmaps to D:\clustered_heatmaps_thursday_0212\ddos_connect_val

Generating 300 windows
Saved 300 heatmaps to D:\clustered_heatmaps_thursday_0212\ddos_publish_val



In [ ]:
#GENERATING TEST

print("Generating test heatmaps")

benign_df = pd.read_csv(r"D:\CiCIoMT\result_flows\Benign_test.pcap_Flow.csv").replace([np.inf, -np.inf], np.nan).fillna(0)
ddos_con_df = pd.read_csv(r"D:\CiCIoMT\result_flows\MQTT-DDoS-Connect_Flood_test.pcap_Flow.csv").replace([np.inf, -np.inf], np.nan).fillna(0)
ddos_pub_df = pd.read_csv(r"D:\CiCIoMT\result_flows\MQTT-DDoS-Publish_Flood_test.pcap_Flow.csv").replace([np.inf, -np.inf], np.nan).fillna(0)


# Benign test
generate_clustered_heatmaps(
    benign_df,
    features=features,
    save_dir=r"D:\clustered_heatmaps_thursday_test",
    label="benign_test",
    n_clusters=20,
    group_size=30,
    scaler=scaler,
    random_sampling=False,
    kmeans_model=benign_kmeans  
)

# DDoS connect test
generate_clustered_heatmaps(
    ddos_con_df,
    features=features,
    save_dir=r"D:\clustered_heatmaps_thursday_test",
    label="ddos_connect_test",
    n_clusters=19,
    group_size=30,
    max_windows=1000,   
    scaler=scaler,
    random_sampling=True,
    kmeans_model=ddos_con_kmeans   
)

# DDoS publish test
generate_clustered_heatmaps(
    ddos_pub_df,
    features=features,
    save_dir=r"D:\clustered_heatmaps_thursday_test",
    label="ddos_publish_test",
    n_clusters=20,
    group_size=30,
    max_windows=1000,   
    scaler=scaler,
    random_sampling=True,
    kmeans_model=ddos_pub_kmeans   
)

Generating test heatmaps
Generating 209 windows
Saved 209 heatmaps to D:\clustered_heatmaps_thursday_test\benign_test

Generating 1000 windows
Saved 1000 heatmaps to D:\clustered_heatmaps_thursday_test\ddos_connect_test

Generating 343 windows
Saved 343 heatmaps to D:\clustered_heatmaps_thursday_test\ddos_publish_test



(MinMaxScaler(),
 MiniBatchKMeans(batch_size=2048, n_clusters=20, n_init=3, random_state=42))